In [0]:
# Databricks Notebook: silver_transform.py
from pyspark.sql.functions import col, when, lit, array
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Configuration widgets
dbutils.widgets.text("catalog_name", "azure-medallion-university-chapters")
dbutils.widgets.text("base_path", "/Volumes")
dbutils.widgets.text("run_id", "")

# Get configuration values
catalog_name = dbutils.widgets.get("catalog_name")
base_path = dbutils.widgets.get("base_path")
run_id = dbutils.widgets.get("run_id")

# Dynamic path construction
bronze_path = f"{base_path}/{catalog_name}/bronze/university_chapters/{run_id}/"
silver_path = f"{base_path}/{catalog_name}/silver/university_chapters"
quarantine_path = f"{base_path}/{catalog_name}/quarantine/university_chapters/{run_id}/"

# Read Bronze data with retry logic for API failures/schema mismatches
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, ArrayType
import time

schema = StructType([
    StructField('features', ArrayType(StructType([
        StructField('attributes', StructType([
            StructField('ChapterID', StringType(), True),
            StructField('University_Chapter', StringType(), True),
            StructField('City', StringType(), True),
            StructField('State', StringType(), True)
        ])),
        StructField('geometry', StructType([
            StructField('x', DoubleType(), True),
            StructField('y', DoubleType(), True)
        ]))
    ])), True)
])

max_retries = 3
retry_delay = 5  # seconds

for attempt in range(max_retries):
    try:
        bronze_df = spark.read.schema(schema).json(bronze_path)
        break
    except Exception as e:
        if attempt < max_retries - 1:
            print(f"Read attempt {attempt+1} failed: {e}. Retrying in {retry_delay} seconds...")
            time.sleep(retry_delay)
        else:
            print(f"Read attempt {attempt+1} failed: {e}. No more retries.")
            raise

# Flatten features
features_df = bronze_df.selectExpr("explode(features) as feature") \
    .select("feature.attributes.*", "feature.geometry.x", "feature.geometry.y")

# Rename + enforce schema
silver_df = features_df.select(
    col("ChapterID").alias("chapter_id"),
    col("University_Chapter").alias("chapter_name"),
    col("City").alias("city"),
    col("State").alias("state"),
    col("x").alias("longitude"),
    col("y").alias("latitude")
)

# Apply DQ-Q1 (invalid coordinates → quarantine)
invalid_coords = (
    (col("longitude").isNull()) |
    (col("latitude").isNull()) |
    (col("longitude") < -180) | (col("longitude") > 180) |
    (col("latitude") < -90) | (col("latitude") > 90)
)

quarantine_df = silver_df.filter(invalid_coords) \
    .withColumn("reason", lit("INVALID_COORDINATES")) \
    .withColumn("ingest_run_id", lit(run_id))

# Apply DQ-W1 (missing/unknown city → warning)
silver_clean_df = silver_df.filter(~invalid_coords) \
    .withColumn(
        "dq_status",
        when(col("city").isNull() | (col("city") == "") | (col("city").rlike("(?i)UNKNOWN")),
             lit("WARNING")).otherwise(lit("OK"))
    ) \
    .withColumn(
        "dq_warnings",
        when(col("dq_status") == "WARNING", array(lit("MISSING_OR_UNKNOWN_CITY"))).otherwise(array())
    )

# Save outputs with exception handling
try:
    silver_clean_df.write.mode("overwrite").parquet(silver_path)
except Exception as e:
    print(f"Failed to write silver data: {e}")
    raise

try:
    quarantine_df.write.mode("overwrite").parquet(quarantine_path)
except Exception as e:
    print(f"Failed to write quarantine data: {e}")
    raise

# Log counts
rows_in = silver_df.count()
rows_quarantined = quarantine_df.count()
rows_warned = silver_clean_df.filter(col("dq_status") == "WARNING").count()
rows_ok = silver_clean_df.filter(col("dq_status") == "OK").count()

print(f"Run {run_id}: In={rows_in}, Quarantined={rows_quarantined}, Warned={rows_warned}, OK={rows_ok}")